# Reproducible Zebrahub pretraining for Biohub cell tracking

This notebook documents the independent-data pretraining run completed on a Quadro RTX 6000. It is deliberately not a Kaggle submission recipe: it uses runtime-loaded Zebrahub crops and lineage labels, records hashes, and keeps the resulting checkpoint separate from competition LB evidence.

The prepared dataset contains 256 train windows and 64 time-disjoint validation windows. The validation interval is separated from training by a 21-frame gap. Check the source terms before redistributing the data.

## 1. Paths and source receipt

In [ ]:
from pathlib import Path
import hashlib, json

PROJECT = Path.cwd()
DATA = PROJECT / 'outputs/exp025_zebrahub_extract/zebrahub_training_set'
RECEIPT = DATA / 'receipt.json'
assert RECEIPT.exists(), RECEIPT
receipt = json.loads(RECEIPT.read_text())
print({k: receipt[k] for k in ('status', 'train_shards', 'valid_shards', 'track_sha256', 'time_split', 'output_shape_zyx')})
print('receipt_sha256 =', hashlib.sha256(RECEIPT.read_bytes()).hexdigest())
assert receipt['status'] == 'PASS'
assert receipt['raw_track_table_deleted'] is True
assert len(list(DATA.glob('train_*.npz'))) == receipt['train_shards']
assert len(list(DATA.glob('valid_*.npz'))) == receipt['valid_shards']

## 2. Reproduce validation and training

The pinned model script verifies every shard against the receipt before materializing windows. On Quadro RTX 6000/PyTorch 2.6, use the math-SDPA runner: the default fused attention path raised `CUDA error: invalid configuration argument` for the large batch-of-short-sequences temporal attention layout. The fallback changes only the attention backend, not the model or loss.

In [ ]:
import os, subprocess, sys

# For a local run, point these variables at the dataset and output directory.
os.environ['BIOHUB_INPUT_ROOT'] = str(PROJECT / 'outputs/exp025_zebrahub_extract')
os.environ['BIOHUB_WORK_ROOT'] = str(PROJECT / 'outputs/research/zebrahub_pretrain_notebook_run')
os.environ['BIOHUB_SUPPORT_ROOT'] = str(PROJECT / 'outputs/exp027_zebrahub_runtime_smoke/tracking_repo')
os.environ['BIOHUB_SKIP_DEP_INSTALL'] = '0'

# Validation only: set BIOHUB_VALIDATE_ONLY=1. Remove it for the 12-epoch run.
os.environ['BIOHUB_VALIDATE_ONLY'] = '1'
cmd = [sys.executable, 'scripts/run_zebrahub_pretrain_math_sdp.py',
       'outputs/exp027_zebrahub_runtime_smoke/pretrain_zebrahub_pinned.py']
print(' '.join(cmd))
# subprocess.run(cmd, check=True, env=os.environ.copy())

## 3. Recorded run result

The RTX6000 run used seed 314159, batch size 2, 12 epochs, and the exact 256/64 shard split. Best external validation selection score was 0.9666307662 (accuracy 0.9998996682, recall 0.9667277598). This is useful pretraining evidence, but it is not a claim of competition-LB improvement.

In [ ]:
run_receipt = json.loads((PROJECT / 'outputs/research/zebrahub_pretrain_20260908/exp026_receipt.json').read_text())
print('status:', run_receipt['status'])
print('best_external_score:', run_receipt['best_external_score'])
print('checkpoint_sha256:', run_receipt['checkpoint_sha256'])
for row in run_receipt['history']:
    print(row['epoch'], row['selection_score'], row['valid_acc'], row['valid_recall'])

## 4. Promotion gate

Do not promote this checkpoint from the external validation score alone. The next honest test is a compute-matched fine-tune on competition training data followed by reciprocal embryo-disjoint LOEO/OOF comparison against the same architecture trained from the competition-only baseline. Only a positive paired result with stable per-embryo behavior can justify an optional clean full-inference submission.